# Reproducing the M33 colliding-flow results

A thin driver over the `exgal_flows` package. It regenerates the three
headline results:

1. **CO sits at colliding flows** - median velocity divergence of CO-bright
   pixels is negative and steepens with CO brightness.
2. **Shear suppresses CO** - the CO-bright fraction falls with shear.
3. **On 50-100 pc scales** - the signal survives smoothing to ~50-100 pc.

All physics lives in the package; this notebook only calls it and plots.
See `FINDINGS.md` for the important aperture (pixels vs. parsecs) caveat.

In [ ]:
import matplotlib.pyplot as plt
from exgal_flows import FlowAnalysis, plotting

DATA = "../data"
ana = FlowAnalysis.from_files(
    mom1_path=f"{DATA}/m33_A+B+C+D+GBT_hi21cm_strict_mom1.fits",
    mom0_path=f"{DATA}/m33_A+B+C+D+GBT_hi21cm_strict_mom0.fits",
    model_path=f"{DATA}/rad.mod.fits",
    co_paths={"hires": f"{DATA}/m33_7m_co21_2p5kms_pbcorr_round.image.max.fits"},
    config_path="../config/m33.yaml",
)
print(f"pixel scale: {ana.pixel_scale_pc:.2f} pc/pix")

## Result 1 - CO sits at colliding flows

In [ ]:
div = ana.result_divergence_of_co(decim=2)
for lab, med, lo, hi in zip(div.labels, div.medians, div.err_low, div.err_high):
    print(f"{lab:10s} median div = {med:+7.2f} [{lo:+6.2f}, {hi:+6.2f}] km/s/kpc")

fig, ax = plt.subplots(figsize=(8, 5))
plotting.plot_divergence_of_co_bright(div, ax=ax)
plt.show()

## Result 2 - shear suppresses CO

In [ ]:
shear = ana.result_co_vs_shear(decim=2)
fig, ax = plt.subplots(figsize=(8, 5))
plotting.plot_co_fraction_vs_gradient(shear, ax=ax, xlabel="|shear| [km/s/pc]")
plt.show()

## Result 3 - the 50-100 pc resolution study

In [ ]:
study = ana.resolution_study(decim=2)
plotting.plot_resolution_study(study, which="divergence")
plt.show()

## Result 4 - how sensitive is CO suppression to shear?

Result 2 shows the CO-bright fraction falls with `|shear|`. Here we quantify
*how much* it falls -- the **sensitivity** `rel_drop = 1 - hi_frac/lo_frac`
(low-shear vs high-shear half of the range) -- and check that it is robust
across four axes: HI epoch, CO-intensity cut, smoothing scale, and
galactocentric radius. Errors are the azimuthal wedge **block bootstrap**
(honest for spatially-correlated maps). Each curve is normalized to its own
low-shear value so the different absolute CO fractions are comparable.

Everything is one package call, `ana.shear_sensitivity(...)`, varying a
single argument per axis. See `FINDINGS.md` for the full tables.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.2, 7.4), sharex=True)
axA, axB, axC, axD = axes.ravel()

# (b) CO-intensity cut, (c) smoothing, (d) radius all use the new `ana`
co_cut = {f"CO>{f:g}x": ana.shear_sensitivity(decim=2, co_threshold=0.6 * f,
                                              n_boot=250)
          for f in ana.config.analysis.co_factor_bins}
smooth = {("native" if fw == 0 else f"{fw:.0f} pc"):
          ana.shear_sensitivity(decim=2, fwhm_pc=fw, n_boot=200)
          for fw in ana.config.analysis.smoothing_fwhm_pc}
radial = {f"{lo}-{hi} pc": ana.shear_sensitivity(decim=2,
                                                 radius_window_pc=(lo, hi), n_boot=250)
          for lo, hi in [(0, 900), (900, 1500), (1500, 2500), (2500, 4000)]}

# (a) epoch: this loads the OLD HI cube (a second ~5 GB read)
ana_old = FlowAnalysis.from_files(
    mom1_path=f"{DATA}/M33_14B_17B_HI_contsub_width_1kms.image.pbcor.GBT_feathered_K.mom1.fits",
    mom0_path=f"{DATA}/M33_14B_17B_HI_contsub_width_1kms.image.pbcor.GBT_feathered_K.mom0.fits",
    model_path=f"{DATA}/rad.mod.fits",
    co_paths={"hires": f"{DATA}/m33_7m_co21_2p5kms_pbcorr_round.image.max.fits"},
    config_path="../config/m33_old.yaml",
)
epoch = {'new (5.3")': ana.shear_sensitivity(decim=2, n_boot=250),
         'old (8.2")': ana_old.shear_sensitivity(decim=2, n_boot=250)}

plotting.plot_shear_sensitivity(epoch, ax=axA,
                                colors=["#1f4e79", "#c1666b"])
plotting.plot_shear_sensitivity(co_cut, ax=axB,
                                colors=["#9ecae1", "#6baed6", "#4292c6", "#1f4e79"])
plotting.plot_shear_sensitivity(smooth, ax=axC, band=False,
                                colors=["#1f4e79", "#8c8c8c", "#6f6f6f", "#525252", "#2f2f2f"])
plotting.plot_shear_sensitivity(radial, ax=axD,
                                colors=["#bdbdbd", "#969696", "#636363", "#1f4e79"])
for ax, t in zip((axA, axB, axC, axD),
                 ["Epoch (CO>2x)", "CO-intensity cut", "Smoothing FWHM", "Radial annulus"]):
    ax.set_title(t)
fig.suptitle("CO-bright fraction is suppressed by velocity shear")
fig.tight_layout()
plt.show()

# the headline sensitivity number
base = epoch['new (5.3")']
print(f"baseline sensitivity (new, CO>2x): rel_drop = {base.rel_drop:.2f} "
      f"[{base.rel_drop_lo:.2f}, {base.rel_drop_hi:.2f}]")